In [2]:
"""
E-Commerce Intelligence Suite
==============================
02_churn_model.py

Was dieses Script macht:
    Trainiert ein Random Forest Modell zur Churn-Vorhersage auf
    Kundenebene. Features werden aus Transaktionsdaten aggregiert.
    Wichtig: recency_days wird NICHT als Feature verwendet, da es
    direkt aus der Churn-Definition abgeleitet ist (Data Leakage).

Business Impact:
    Anstatt zu warten bis ein Kunde abspringt, kann ein Shop 60 Tage
    vorher sehen wer gefaehrdet ist -- und gezielt reagieren mit
    Win-Back Kampagnen, Rabatten oder personalisierten Emails.
    Bei 451 At-Risk Kunden und Ø 976 GBP Umsatz = ~88k GBP Potenzial.

Modell-Entscheidung:
    Random Forest weil:
    - Robust gegenueber Ausreissern (hohe Bestellwerte)
    - Gibt Feature Importance direkt aus (erklaerbar fuer Kunden)
    - Kein Scaling noetig
    - 70% Accuracy bei synthetischen Daten -- realistisch und ehrlich

Voraussetzungen:
    conda activate ecommerce-suite
    pip install -r requirements.txt

Ausfuehren:
    python python/02_churn_model.py
"""

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score
)

# ------------------------------------------------------------------ #
# Konfiguration
# ------------------------------------------------------------------ #
RANDOM_SEED  = 42
INPUT_FILE   = r'C:\Users\Johan\Documents\Projects_for_data_structure\sample_projects\ecommerce-intelligence-suite\data\data_enriched_fixed.csv'
CHURN_DAYS   = 60   # Muss identisch mit 01_data_generation.py sein
TEST_SIZE    = 0.2

np.random.seed(RANDOM_SEED)

# ------------------------------------------------------------------ #
# 1. Daten laden
# ------------------------------------------------------------------ #
print('Lade Daten...')
df = pd.read_csv(INPUT_FILE)
df['invoicedate'] = pd.to_datetime(df['invoicedate'])
print(f'  {len(df):,} Transaktionen geladen')

# ------------------------------------------------------------------ #
# 2. Train/Test Split auf Zeitbasis
# ------------------------------------------------------------------ #
# Wichtig: Wir trainieren nur auf Daten VOR dem Cutoff.
# Das simuliert eine echte Prediction: "Wer wird in den naechsten
# 60 Tagen nicht mehr kaufen?" -- ohne in die Zukunft zu schauen.
max_date = df['invoicedate'].max()
cutoff   = max_date - pd.Timedelta(days=CHURN_DAYS)
df_train = df[df['invoicedate'] <= cutoff].copy()

print(f'  Trainings-Zeitraum: {df_train["invoicedate"].min().date()} bis {cutoff.date()}')
print(f'  {len(df_train):,} Transaktionen im Trainingsfenster')

# ------------------------------------------------------------------ #
# 3. Feature Engineering auf Kundenebene
# ------------------------------------------------------------------ #
# Jede Zeile = ein Kunde mit aggregierten Features aus seinen Kaeufen
print('\nFeature Engineering...')

customers = (
    df_train[df_train['customer_id'] != 0]
    .groupby('customer_id')
    .agg(
        # Kaufverhalten
        frequency       =('invoiceno',     'nunique'),       # Anzahl Bestellungen
        monetary        =('revenue',        'sum'),           # Gesamtumsatz
        avg_order_value =('revenue',        'mean'),          # Ø Bestellwert
        total_items     =('quantity',       'sum'),           # Anzahl Artikel
        unique_products =('stockcode',      'nunique'),       # Produktvielfalt

        # Email Engagement
        email_open_rate =('email_opened',   'mean'),          # Anteil geoeffneter Emails
        email_click_rate=('email_clicked',  'mean'),          # Anteil geklickter Emails

        # Traffic Herkunft
        pct_email       =('utm_source', lambda x: (x == 'email').mean()),
        pct_google      =('utm_source', lambda x: (x == 'google').mean()),

        # Zielvariable
        churned         =('churned',        'max')
    )
    .reset_index()
)

print(f'  {len(customers):,} Kunden mit Features')
print(f'  Churn Rate: {customers["churned"].mean() * 100:.1f}%')
print(f'  Features: {[c for c in customers.columns if c not in ["customer_id", "churned"]]}')

# ------------------------------------------------------------------ #
# 4. Train / Test Split
# ------------------------------------------------------------------ #
X = customers.drop(columns=['customer_id', 'churned'])
y = customers['churned']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_SEED,
    stratify=y   # Gleiche Churn-Verteilung in Train und Test
)

print(f'\nTrain: {len(X_train):,} · Test: {len(X_test):,}')

# ------------------------------------------------------------------ #
# 5. Modell trainieren
# ------------------------------------------------------------------ #
print('\nTrainiere Random Forest...')

rf = RandomForestClassifier(
    n_estimators=100,
    random_state=RANDOM_SEED,
    class_weight='balanced',  # Ausgleich falls Churn/Non-Churn ungleich
    n_jobs=-1
)
rf.fit(X_train, y_train)
print('  Fertig.')

# ------------------------------------------------------------------ #
# 6. Evaluation
# ------------------------------------------------------------------ #
y_pred      = rf.predict(X_test)
y_pred_prob = rf.predict_proba(X_test)[:, 1]

print('\n--- ERGEBNISSE ---')
print(f'Accuracy:  {(y_pred == y_test).mean() * 100:.1f}%')
print(f'ROC-AUC:   {roc_auc_score(y_test, y_pred_prob):.3f}')

print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=['Kein Churn', 'Churn']))

print('Confusion Matrix:')
cm = confusion_matrix(y_test, y_pred)
print(f'  True Negatives  (korrekt: bleibt):    {cm[0][0]}')
print(f'  False Positives (falsch: wird churnen): {cm[0][1]}')
print(f'  False Negatives (verpasst: ist weg):  {cm[1][0]}')
print(f'  True Positives  (korrekt: churnt):    {cm[1][1]}')

# ------------------------------------------------------------------ #
# 7. Feature Importance
# ------------------------------------------------------------------ #
print('\nFeature Importance (absteigend):')
importance = sorted(
    zip(X.columns, rf.feature_importances_),
    key=lambda x: -x[1]
)
for feat, imp in importance:
    bar = '█' * int(imp * 50)
    print(f'  {feat:<20} {imp:.4f}  {bar}')

# ------------------------------------------------------------------ #
# 8. Churn-Wahrscheinlichkeit pro Kunde exportieren
# ------------------------------------------------------------------ #
results = X_test.copy()
results['customer_id']      = customers.loc[X_test.index, 'customer_id'].values
results['churned_actual']   = y_test.values
results['churn_probability'] = y_pred_prob
results['churn_predicted']  = y_pred
results = results.sort_values('churn_probability', ascending=False)

output_path = r'C:\Users\Johan\Documents\Projects_for_data_structure\sample_projects\ecommerce-intelligence-suite\data\churn_predictions.csv'
results[['customer_id', 'churn_probability', 'churn_predicted', 'churned_actual']].to_csv(
    output_path, index=False
)
print(f'\nPredictions gespeichert: {output_path}')
print(f'Top 5 Churn-Risiko Kunden:')
print(results[['customer_id', 'churn_probability']].head())

Lade Daten...
  300,000 Transaktionen geladen
  Trainings-Zeitraum: 2010-12-01 bis 2011-06-13
  225,043 Transaktionen im Trainingsfenster

Feature Engineering...
  2,878 Kunden mit Features
  Churn Rate: 58.5%
  Features: ['frequency', 'monetary', 'avg_order_value', 'total_items', 'unique_products', 'email_open_rate', 'email_click_rate', 'pct_email', 'pct_google']

Train: 2,302 · Test: 576

Trainiere Random Forest...
  Fertig.

--- ERGEBNISSE ---
Accuracy:  67.7%
ROC-AUC:   0.695

Classification Report:
              precision    recall  f1-score   support

  Kein Churn       0.64      0.49      0.56       239
       Churn       0.69      0.81      0.75       337

    accuracy                           0.68       576
   macro avg       0.67      0.65      0.65       576
weighted avg       0.67      0.68      0.67       576

Confusion Matrix:
  True Negatives  (korrekt: bleibt):    118
  False Positives (falsch: wird churnen): 121
  False Negatives (verpasst: ist weg):  65
  True Positi